# Entrenamiento supervisado de Karen — Kaggle

Este notebook prepara y ejecuta un primer ajuste QLoRA con Unsloth. **No crea un modelo fundacional desde cero** y no usa APIs de ChatGPT, Gemini ni Claude. Descarga pesos abiertos una vez, los guarda en la sesión y el script de Karen entrena en modo local/offline.

Antes de ejecutar: activa una GPU en Settings, añade como Dataset privado un `raw.jsonl` con ejemplos revisados y aprobados, y no subas conversaciones privadas, contraseñas ni datos sensibles. Los 40 ejemplos candidatos actuales siguen pendientes y no se usarán automáticamente.


In [ ]:
from pathlib import Path
import os, subprocess, json, glob, shutil

REPO_URL = "https://github.com/Turnesito130/karen-IA.git"
BRANCH = "arquitectura/cerebro-propio-0.1"
MODEL_ID = "unsloth/SmolLM3-3B"  # candidato provisional; Apache 2.0; multilingüe con español
ROOT = Path("/kaggle/working/karen-IA")
MODEL_DIR = Path("/kaggle/working/base-model")
RUN_DIR = Path("/kaggle/working/karen-run")
print("Karen branch:", BRANCH)
print("Candidate model:", MODEL_ID)


In [ ]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU CUDA activa. Activa GPU en los ajustes del Notebook; no se iniciará QLoRA en CPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM total (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    print("Repo ya presente:", ROOT)
os.chdir(ROOT)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## Instalar herramientas

Unsloth actualiza sus combinaciones compatibles de PyTorch/CUDA. Si su guía oficial indica un comando específico para el runtime de Kaggle, utiliza ese comando. La instalación de abajo es el punto de partida genérico; el diagnóstico posterior decide si el entorno está listo.


In [ ]:
%pip install --upgrade unsloth
%pip install -r training/requirements.txt
%pip install --upgrade "transformers>=4.53.0" "accelerate>=1.0" "datasets>=3.0,<5" "trl>=0.15" "peft>=0.13"


In [ ]:
subprocess.run(["python", "training/check_environment.py"], check=True)
print("Entorno comprobado. Si el diagnóstico muestra incompatibilidades, detente y corrige la instalación antes de entrenar.")


## Datos revisados y aprobados

Añade a Kaggle un Dataset privado que contenga un único `raw.jsonl`. Todas las líneas deben tener `approved: true`, `instruction`, `output` y una fuente. El notebook se detiene si no encuentra exactamente un archivo o si la auditoría informa problemas.


In [ ]:
matches = glob.glob("/kaggle/input/**/raw.jsonl", recursive=True)
if len(matches) != 1:
    raise FileNotFoundError(f"Se esperaba exactamente un raw.jsonl en los Datasets adjuntos; encontrados: {matches}")
raw_source = Path(matches[0])
raw_target = ROOT / "training/datasets/raw.jsonl"
raw_target.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(raw_source, raw_target)
print("Dataset privado copiado a la sesión:", raw_target)
audit_path = ROOT / "training/datasets/audit-kaggle.json"
subprocess.run(["python", "training/audit_dataset.py", "--input", str(raw_target), "--output", str(audit_path)], check=True)
audit = json.loads(audit_path.read_text(encoding="utf-8"))
print(json.dumps(audit, ensure_ascii=False, indent=2))
if audit.get("issues") or audit.get("near_duplicates"):
    raise RuntimeError("La auditoría detectó problemas. Corrige el Dataset y vuelve a adjuntarlo; no se entrenará.")
approved_count = sum(1 for line in raw_target.read_text(encoding="utf-8").splitlines() if line.strip() and json.loads(line).get("approved") is True)
if approved_count < 10:
    raise RuntimeError(f"Solo hay {approved_count} ejemplos aprobados; se necesitan al menos 10 para crear splits.")
subprocess.run(["python", "training/prepare_dataset.py", "--input", str(raw_target), "--output-dir", "training/datasets/prepared"], check=True)
print("Dataset preparado:", json.loads(Path("training/datasets/prepared/report.json").read_text(encoding="utf-8"))["counts"])


## Descargar pesos abiertos del candidato

El candidato provisional es `unsloth/SmolLM3-3B`: 3B de parámetros, licencia Apache 2.0 y soporte declarado para español. Esto no implica que vaya a caber en cualquier GPU gratuita; el entrenamiento se detendrá si CUDA o la memoria no son suficientes. Revisa la ficha y la licencia antes de continuar: https://huggingface.co/unsloth/SmolLM3-3B


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id=MODEL_ID, local_dir=str(MODEL_DIR))
if not any(MODEL_DIR.iterdir()):
    raise RuntimeError("La descarga del modelo no produjo archivos locales.")
print("Modelo descargado localmente:", MODEL_DIR)


In [ ]:
subprocess.run([
    "python", "training/train_unsloth.py",
    "--model-path", str(MODEL_DIR),
    "--base-model-id", MODEL_ID,
    "--dataset-dir", "training/datasets/prepared",
    "--output-dir", str(RUN_DIR),
    "--max-seq-length", "1024",
    "--rank", "8",
    "--epochs", "1",
    "--batch-size", "1",
    "--grad-accumulation", "4",
    "--offline"
], check=True)
print("Entrenamiento terminado. El adaptador aún no está conectado a la aplicación Karen.")


## Evaluación obligatoria

Se generan respuestas del modelo base y del modelo adaptado sobre el mismo conjunto `test`. No uses esas respuestas para entrenar. La revisión y la puntuación deben ser humanas.


In [ ]:
evaluation_dir = ROOT / "training/evaluation/kaggle-first-run"
evaluation_dir.mkdir(parents=True, exist_ok=True)
test_file = "training/datasets/prepared/test.jsonl"
subprocess.run(["python", "training/evaluate_model.py", "--model-path", str(MODEL_DIR), "--test", test_file, "--output", str(evaluation_dir / "base.jsonl")], check=True)
subprocess.run(["python", "training/evaluate_model.py", "--model-path", str(MODEL_DIR), "--adapter", str(RUN_DIR / "adapter"), "--test", test_file, "--output", str(evaluation_dir / "adapted.jsonl")], check=True)
print("Revisa manualmente:", evaluation_dir)


In [ ]:
artifact = Path("/kaggle/working/karen-first-training-artifacts.zip")
shutil.make_archive(str(artifact.with_suffix("")), "zip", "/kaggle/working", "karen-run")
print("Artefacto preparado:", artifact)
print("Descárgalo desde Output del Notebook antes de que expire la sesión. No lo subas a un repositorio público.")
